# Lab 2 Writeup

## Prelab Question 1

Prelab Question 1: What is the latency of an FFMA instruction?

**Answer:**

The latency of an FFMA instruction is 4 cycles. To guarantee that the multiply and add are actually fused, I chose to use the `ffma` instruction directly with [fmaf](https://docs.nvidia.com/cuda/cuda-math-api/cuda_math_api/group__CUDA__MATH__SINGLE.html#_CPPv44fmaffff).

Our kernel contains chained dependent FMA operations
```
x = fmaf(x, x, x);
x = fmaf(x, x, x);
x = fmaf(x, x, x);
x = fmaf(x, x, x);
x = fmaf(x, x, x);
x = fmaf(x, x, x);
``` 
To execute the program, we run the cmd
```bash
nvcc fma_latency.cu -o fma_latency -lineinfo -arch=native && cuobjdump --dump-sass fma_latency > sass_output.txt && ./fma_latency
```

We inspect the SASS code in `sass_output.txt`
```
/*0090*/                   FFMA R0, R0, R0, R0 ;             /* 0x0000000000007223 */
                                                                /* 0x004fc80000000000 */
/*00a0*/                   FFMA R0, R0, R0, R0 ;             /* 0x0000000000007223 */
                                                                /* 0x000fc80000000000 */
/*00b0*/                   FFMA R0, R0, R0, R0 ;             /* 0x0000000000007223 */
                                                                /* 0x000fc80000000000 */
/*00c0*/                   FFMA R0, R0, R0, R0 ;             /* 0x0000000000007223 */
                                                                /* 0x000fc80000000000 */
/*00d0*/                   FFMA R0, R0, R0, R0 ;             /* 0x0000000000007223 */
                                                                /* 0x000fc80000000000 */
/*00e0*/                   FFMA R13, R0, R0, R0 ;            /* 0x00000000000d7223 */
```
Thus confirms we actually issue FFMA instructions.


I noticed that if we change the number of FFMA instructions, we get different number of clock cycles

|# FMA | clock cycles | 
|------|--------------|
| 1    | 2 |
| 2 | 6 |
| 3 | 10 |
| 4 | 14 |
| 5 | 18 | 
| 6 | 22 |

Each additional FMA instruction after the first FMA takes four clock cycles. However, when we only have one FMA, it only takes two clock cycles. I'm not quite sure why this is. But it is a constant overhead.

I believe the first FMA takes two clock cycles because we measure the `end_time` after the last FMA is issued but before the last FMA completes. So we record the end time two clock cycles too early, two cycles before the last FMA actually completes. This makes sense as there is no dependencies between the instructions `x = fmaf(x, x, x);` and `end_time = clock_cycle();` so they can be overlapped.

(I noticed that the last FFMA instruction always writes to register R13 whereas the other FFMA instructions always write to register R0. But I don't think this should have any impact on the timing.)



Some quick googling also confirms that fp32 FFMA takes around 4 clock cycles on my hardware, a 5080 ([Dissecting the SM_120 Microarchitecture
](https://zartbot.github.io/micro_arch/nvidia/sm_120/paper.html) section 6.2.)


## Prelab Question 2

In fma-latency.cu, complete the fma_latency_interleaved function using a sequence of explicitly interleaved FMA operations designed to exploit ILP. What latency do you observe while running this sequence of operations? Does the observed number contradict your observations from before?

**Answer:**

We explicitly interleave the FMA operations, alternating between `x` and `y`. For example:
```cu
x = fmaf(x, x, x);
y = fmaf(y, y, y);
x = fmaf(x, x, x);
y = fmaf(y, y, y);
x = fmaf(x, x, x);
y = fmaf(y, y, y);
```

We can compare the time (in clock cycles) for performing the non-interleaved and interleaved FMAs.

|# FMA instructions | non-interleaved time | interleaved time | 
|------|--------------|-----|
| 2    | 6 | 4 |
| 4 | 14 | 8 |
| 6 | 22 | 12 |

When we have two FMA instructions, it takes 6 cycles for non-interleaved FMAs: 4 cycles for the first FMA and 2 cycles for the second FMA. (Remember, the last FMA takes 2 cycles instead of 4 cycles because the clock records when we finish issuing the last FMA instruction, not when the last FMA instruction finishes executing.) But when we have interleaved FMA instructions, we perform both FMAs at the same time and each one of them should take 2 cycles since the clock records when the last FMA instruction is finished being issued, not finished executing. But, oddly enough, we actually see it takes 4 cycles. What's going on?